## Escenario 2: Equipo colaborativo con MLflow Tracking Server local

MLflow setup:
- Tracking server: **sí** (local)
- Backend store: **SQLite** (archivo local)
- Artifacts store: **filesystem local**
- Model Registry: **disponible** (útil para aprender el flujo), pero para producción se recomienda backend DB + artifact store remoto (ver Scenario 3)

Antes de ejecutar este notebook, inicia el servidor MLflow (desde la raíz del proyecto):

```bash
mlflow server \
  --host 127.0.0.1 \
  --port 5000 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns
```

Luego abre la UI en: http://127.0.0.1:5000

## Configuracion del tracking URI

Conecta el cliente MLflow al servidor local en el puerto 5000.


In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

Confirma que la conexión al servidor se estableció correctamente.

## Listado de Experimentos Disponibles

In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1758647875569, experiment_id='0', last_update_time=1758647875569, lifecycle_stage='active', name='Default', tags={}>]

## Ejecución del Experimento

In [ ]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_experiment("iris-server-local")

with mlflow.start_run(run_name="logreg_baseline"):
    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42, "max_iter": 200}
    mlflow.log_params(params)

    mlflow.set_tags(
        {
            "scenario": "2_local_tracking_server",
            "developer": "Maria Durango",
            "module": "mlops_tracking",
            "model_family": "logistic_regression",
            "dataset": "iris",
        }
    )

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)

    mlflow.log_metric("accuracy", float(accuracy_score(y, y_pred)))
    mlflow.log_metric("f1_macro", float(f1_score(y, y_pred, average="macro")))

    preds_path = "predictions_logreg.csv"
    pd.DataFrame({"y_true": y, "y_pred": y_pred}).to_csv(preds_path, index=False)
    mlflow.log_artifact(preds_path)

    model_info = mlflow.sklearn.log_model(
        lr,
        name="model",
        input_example=X[[0]],
        registered_model_name="iris-classifier",
    )

    print({"model_uri": model_info.model_uri, "artifact_uri": mlflow.get_artifact_uri()})

2025/09/23 12:22:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/23 12:22:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


default artifacts URI: 'mlflow-artifacts:/1/302f649e6f4d4844a4df182aa5eaef94/artifacts'
🏃 View run iris_classifier at: http://127.0.0.1:5000/#/experiments/1/runs/302f649e6f4d4844a4df182aa5eaef94
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Resultados de la Ejecución

La ejecución genera:
- Un URI único para los artifacts
- Enlaces directos al servidor MLflow para visualizar:
    - La ejecución específica
    - El experimento completo

Ahora debería aparecer:
- "Default" (creado automáticamente)
- "iris-server-local" (nuestro experimento)

In [7]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1758648004468, experiment_id='1', last_update_time=1758648004468, lifecycle_stage='active', name='iris-classifier', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1758647875569, experiment_id='0', last_update_time=1758647875569, lifecycle_stage='active', name='Default', tags={}>]

### Interactuando con el model registry

Crea un cliente para interactuar con el servidor MLflow.

In [8]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

Listemos los modelos registrados actualmente (si ya corriste el notebook antes, puede que veas múltiples versiones):

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())
client.search_registered_models()

[]

## Model Registry: versiones y aliases (flujo recomendado)

En este escenario ya registramos el modelo usando `mlflow.sklearn.log_model(..., registered_model_name=...)`.

Ahora veremos cómo:
- listar versiones disponibles
- asignar un alias (por ejemplo `champion` / `candidate`)

> Nota: el uso de *stages* existe, pero MLflow recomienda cada vez más usar **aliases** para consumir modelos de forma estable.

In [ ]:
model_name = "iris-classifier"

latest_versions = client.search_model_versions(f"name='{model_name}'")

[(mv.version, mv.current_stage, mv.run_id) for mv in latest_versions]

In [ ]:
# Elegimos la versión más reciente (por creation timestamp)
latest_mv = max(latest_versions, key=lambda mv: int(mv.creation_timestamp))
latest_version = latest_mv.version

# Asignamos aliases típicos
client.set_registered_model_alias(model_name, "candidate", latest_version)
client.set_registered_model_alias(model_name, "champion", latest_version)

client.get_registered_model(model_name).aliases

<RunInfo: artifact_uri='mlflow-artifacts:/1/302f649e6f4d4844a4df182aa5eaef94/artifacts', end_time=1758648126929, experiment_id='1', lifecycle_stage='active', run_id='302f649e6f4d4844a4df182aa5eaef94', run_name='iris_classifier', start_time=1758648124664, status='FINISHED', user_id='mdurango'>

In [ ]:
# (Opcional) Stage: requiere que el servidor tenga backend store (SQLite/Postgres/etc.)
# y el model registry habilitado. En local suele funcionar.

client.transition_model_version_stage(
    name=model_name,
    version=latest_version,
    stage="Staging",
    archive_existing_versions=False,
)

client.get_model_version(name=model_name, version=latest_version).current_stage

Registered model 'iris-classifier' already exists. Creating a new version of this model...
2025/09/23 12:30:30 WARNING mlflow.tracking._model_registry.fluent: Run with id 302f649e6f4d4844a4df182aa5eaef94 has no artifacts at artifact path 'models', registering model based on models:/m-c88109816e9c49b79924c19ba3f957e7 instead
2025/09/23 12:30:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 2
Created version '2' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1758648630081, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1758648630081, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='302f649e6f4d4844a4df182aa5eaef94', run_link='', source='models:/m-c88109816e9c49b79924c19ba3f957e7', status='READY', status_message=None, tags={}, user_id='', version='2'>

## **Model Registry vs Tracking: Diferencias Clave**

### **MLflow Tracking**
- **Propósito**: Seguimiento de experimentos y ejecuciones individuales
- **Almacena**: Parámetros, métricas, artifacts y metadatos de cada run
- **Uso**: Durante el desarrollo y experimentación
- **Organización**: Por experimentos y ejecuciones
- **Acceso**: A través de la API de MLflow o interfaz web

### **Model Registry**
- **Propósito**: Gestión del ciclo de vida completo de modelos en producción
- **Almacena**: Versiones de modelos, stages (Staging, Production), transiciones
- **Uso**: Para despliegue, versionado y gestión operacional
- **Organización**: Por nombre de modelo y versiones
- **Acceso**: A través del Model Registry API


## **Ventaja Principal del Model Registry**

**El Model Registry va más allá del tracking al proporcionar:**

1. **Versionado de Modelos**: Mantiene un historial completo de todas las versiones
2. **Stages de Despliegue**: Controla qué versión está en Staging vs Production
3. **Aprobaciones**: Permite flujos de trabajo para promocionar modelos
4. **Trazabilidad**: Conecta cada versión con su experimento original
5. **Operaciones**: Facilita rollbacks, comparaciones y auditorías

**En resumen**: Tracking es para experimentar, Registry es para producir y mantener modelos en el mundo real.

-----------

## **Diferencias Clave con el Escenario 1**

### **Arquitectura**
- **Escenario 1**: MLflow local con archivos
- **Escenario 2**: Servidor MLflow con base de datos SQLite

### **Colaboración**
- **Escenario 1**: Uso individual
- **Escenario 2**: Múltiples usuarios pueden acceder al mismo servidor

### **Persistencia**
- **Escenario 1**: Datos almacenados en archivos locales
- **Escenario 2**: Metadatos en base de datos SQLite, artifacts en sistema de archivos

### **Model Registry**
- **Escenario 1**: No disponible
- **Escenario 2**: Completamente funcional para gestión de modelos

---

## **Ventajas del Escenario 2**

1. **Colaboración**: Múltiples científicos de datos pueden trabajar en el mismo servidor
2. **Persistencia**: Los metadatos se mantienen en una base de datos estructurada
3. **Escalabilidad**: Fácil de migrar a bases de datos más robustas (PostgreSQL, MySQL)
4. **Model Registry**: Gestión completa del ciclo de vida de los modelos
5. **Acceso Web**: Interfaz web disponible en http://127.0.0.1:5000

---

## **Casos de Uso Apropiados**

- **Equipos pequeños**: 2-10 científicos de datos
- **Desarrollo local**: Entorno de desarrollo y testing
- **Prototipado**: Validación de conceptos antes de producción
- **Educación**: Cursos y talleres de MLOps
- **Investigación**: Experimentos colaborativos en academia